# 04 — Pipeline completo: transação → XGBoost → SHAP → RAG → explicação

Demonstra as três camadas da arquitetura aprovada sobre **10 transações reais do conjunto de teste**, classificadas como suspeitas pelo modelo.

## Antes de executar

Este notebook depende de três artefatos que **não estão no repositório**, porque são grandes ou derivados:

| Artefato | Como obter |
|---|---|
| `models/xgboost/` | `python scripts/tunar_xgboost.py` |
| `data/rag/index/` | pipeline de construção da base vetorial (Pessoa 2) |
| Cliente do LLM | `{{PREENCHER}}` — ver seção 5 |

Se algum faltar, as células correspondentes falham com mensagem explicando o que configurar. Isso é proposital: um notebook que produz resultado plausível com componente ausente é pior que um que falha.

## 1. Carregar dados e reproduzir o mesmo corte temporal

O corte precisa ser **idêntico** ao do treino, senão as transações "de teste" podem ter sido vistas pelo modelo. Por isso o notebook repete exatamente a mesma sequência do notebook 02, em vez de sortear linhas.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

RAIZ = Path().resolve().parent
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from src.data_loader import carregar_dados
from src.features.pix_features import criar_features_pix
from src.features.preprocessor import (COLUNA_ALVO, dividir_temporal,
                                       reduzir_precisao)

# None = dataset completo. Para uma execução rápida de verificação, use 200_000.
N_LINHAS = None

dados = carregar_dados(nrows=N_LINHAS)
df = reduzir_precisao(criar_features_pix(dados[2]))
del dados

treino, validacao, teste = dividir_temporal(df)
X_teste, y_teste = teste.drop(columns=[COLUNA_ALVO]), teste[COLUNA_ALVO]
X_treino = treino.drop(columns=[COLUNA_ALVO])
print(f'teste: {len(X_teste):,} transações | {int(y_teste.sum()):,} fraudes'.replace(',', '.'))

## 2. Montar o pipeline

O modelo e o pré-processador vêm do mesmo artefato — o pré-processador **não é opcional**: o modelo espera a matriz de 460 colunas que ele produz, e aplicá-lo a dados crus devolve números errados sem levantar exceção.

A matriz de fundo do SHAP são 500 linhas amostradas **somente do treino**, com seed registrada, conforme o protocolo em `reports/pessoa_2/julho/04_metodologia_shap.md`.

In [ ]:
from src.models.persistencia import carregar
from src.pipeline import construir_pipeline
from src.rag.retriever import RecuperadorDocumentos

SEMENTE = 42
TAMANHO_FUNDO = 500

preprocessador, _, manifesto = carregar(RAIZ / 'models' / 'xgboost')
print('modelo:', manifesto['tipo_do_modelo'], '| gerado em', manifesto['gerado_em'])

rng = np.random.default_rng(SEMENTE)
M_treino = preprocessador.transform(X_treino)
fundo = M_treino[rng.choice(len(M_treino), TAMANHO_FUNDO, replace=False)]
del M_treino

In [ ]:
# O índice vetorial é opcional: sem ele o pipeline roda sem a camada de
# recuperação, e a explicação sai apenas com os fatores do SHAP.
try:
    recuperador = RecuperadorDocumentos.a_partir_do_disco()
    print('índice vetorial carregado:', len(recuperador.indice.documents), 'documentos')
except Exception as erro:
    recuperador = None
    print('sem índice vetorial:', type(erro).__name__, '—', erro)

## 3. Configurar o cliente do LLM

A dupla decidiu em julho usar **Claude Haiku 4.5** como principal e **Ollama local** como alternativa, mas o cliente ainda não foi implementado no projeto.

Enquanto isso, o notebook usa um cliente que **devolve o próprio prompt** em vez de inventar uma explicação. Assim é possível inspecionar exatamente o que seria enviado ao modelo — que é a parte auditável — sem fabricar texto que pareceria resultado.

In [ ]:
def cliente_inspecao(prompt: str) -> str:
    """{{PREENCHER}}: trocar pelo cliente real do LLM.

    Devolve o prompt em vez de uma explicação inventada: o que está sob
    verificação aqui é o conteúdo enviado ao modelo.
    """
    return prompt


# {{PREENCHER}}: substituir por Claude Haiku 4.5 ou Ollama quando o cliente existir.
CLIENTE_LLM = cliente_inspecao

pipeline = construir_pipeline(
    diretorio_modelo=RAIZ / 'models' / 'xgboost',
    matriz_fundo=fundo,
    recuperador=recuperador,
    cliente_llm=CLIENTE_LLM,
)
print('pipeline pronto')

## 4. Selecionar 10 transações sinalizadas pelo modelo

As transações são escolhidas entre as que o **modelo classificou como suspeitas** no conjunto de teste, não entre as rotuladas como fraude. A diferença importa: o protótipo explica a decisão do modelo, e um falso positivo é tão explicável quanto um acerto — e, na prática, mais interessante de examinar.

In [ ]:
N_EXEMPLOS = 10

M_teste = preprocessador.transform(X_teste)
probabilidades = pipeline.modelo.predict_proba(M_teste)[:, 1]
del M_teste

posicoes_sinalizadas = np.flatnonzero(probabilidades >= pipeline.limiar)
print(f'{len(posicoes_sinalizadas):,} transações sinalizadas'.replace(',', '.'))

# As de maior probabilidade: são as que o modelo considera mais claras.
posicoes = posicoes_sinalizadas[np.argsort(probabilidades[posicoes_sinalizadas])[::-1]][:N_EXEMPLOS]

resumo_selecao = pd.DataFrame({
    'posicao': posicoes,
    'probabilidade': probabilidades[posicoes],
    'rotulo_real': y_teste.iloc[posicoes].to_numpy(),
})
resumo_selecao

## 5. Executar o pipeline nas 10 transações

In [ ]:
resultados = []
for posicao in posicoes:
    transacao = X_teste.iloc[[int(posicao)]]
    resultado = pipeline.processar(transacao)
    resultado['posicao'] = int(posicao)
    resultado['rotulo_real'] = int(y_teste.iloc[int(posicao)])
    resultados.append(resultado)

print(f'{len(resultados)} transações processadas')

## 6. Tabela consolidada

Uma linha por transação, com o que o TCC precisa citar: decisão, probabilidade, os três fatores de maior influência e se o modelo acertou.

In [ ]:
tabela = pd.DataFrame([{
    'posição': r['posicao'],
    'predição': r['predicao'],
    'probabilidade': round(r['probabilidade'], 4),
    'rótulo real': r['rotulo_real'],
    'acertou': 'sim' if (r['rotulo_real'] == 1) else 'não (falso positivo)',
    'top-3 SHAP': ', '.join(f"{f['feature']} ({f['contribuicao']:+.3f})" for f in r['fatores_shap']),
    'documentos': len(r['documentos_recuperados']),
} for r in resultados])

pd.set_option('display.max_colwidth', 90)
tabela

## 7. Detalhamento de uma transação

O caso completo, como apareceria na interface de demonstração.

In [ ]:
exemplo = resultados[0]

print('POSIÇÃO:', exemplo['posicao'])
print('DECISÃO :', exemplo['predicao'], f"(probabilidade {exemplo['probabilidade']:.1%}, limiar {exemplo['limiar']})")
print('RÓTULO  :', 'fraude' if exemplo['rotulo_real'] else 'legítima')

print('\nFATORES DE MAIOR INFLUÊNCIA (SHAP)')
for posicao, fator in enumerate(exemplo['fatores_shap'], start=1):
    print(f"  {posicao}. {fator['feature']:<40} {fator['contribuicao']:+.4f}  ({fator['direcao']})")

print('\nDOCUMENTOS RECUPERADOS')
if exemplo['documentos_recuperados']:
    for posicao, documento in enumerate(exemplo['documentos_recuperados'], start=1):
        print(f"  [{posicao}] {documento['fonte']}  (score {documento['score']:.3f})")
        print(f"      {documento['trecho'][:160]}")
else:
    print('  (nenhum — índice vetorial indisponível)')

print('\nCONTEÚDO ENVIADO AO LLM')
print(exemplo['explicacao_rag'][:1500])

## 8. Leitura dos resultados

**Contribuição do SHAP é influência sobre a decisão do modelo, não causa da fraude.** Uma variável com contribuição alta indica que o modelo a usou para chegar àquela probabilidade — não que ela explique por que a fraude ocorreu.

**Colunas anônimas continuam anônimas.** Se `C13` ou `V257` aparecerem entre os fatores mais influentes, a afirmação possível é "esta coluna teve a maior contribuição para a decisão". O IEEE-CIS não publica o que elas representam, e atribuir significado seria inventar semântica.

**Falsos positivos são resultado, não falha da demonstração.** No conjunto de teste, o baseline marcava cerca de 8 em cada 10 acusações incorretamente. Encontrar falsos positivos entre os exemplos é o comportamento esperado do sistema e deve ser discutido no Capítulo 4, não escondido.

**A explicação acima é o prompt, não uma resposta de modelo.** Enquanto o cliente do LLM for `{{PREENCHER}}`, nenhum texto gerado aparece aqui — o que se inspeciona é a evidência que seria enviada.

## Próximos passos

- implementar o cliente do LLM (Claude Haiku 4.5 ou Ollama) e reexecutar;
- avaliar a qualidade das explicações: relevância dos documentos, coerência com o SHAP e ausência de alucinações (`m4_p2_2`);
- documentar exemplos selecionados para o Capítulo 4 (`m4_p2_4`).